# 第 17 章互动实验：合成相机标定

用已知真值的合成相机检验你的标定实现，观察噪声、视图数和覆盖范围对精度的影响。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "17_相机标定"
ASSIGNMENT = "17-1_张正友标定.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
def rodrigues(r):
    t = np.linalg.norm(r)
    if t < 1e-12: return np.eye(3)
    k = r / t; K_ = np.array([[0, -k[2], k[1]], [k[2], 0, -k[0]], [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(t) * K_ + (1 - np.cos(t)) * K_ @ K_

K_TRUE = np.array([[820.0, 0, 318.0], [0, 790.0, 247.0], [0, 0, 1]])
xs, ys = np.meshgrid(np.arange(9) * 15.0, np.arange(6) * 15.0)
BOARD = np.stack([xs.ravel(), ys.ravel()], 1)
BOARD3 = np.hstack([BOARD, np.zeros((len(BOARD), 1))])

def project(K, R, t, P):
    pc = P @ R.T + t; uv = pc @ K.T; return uv[:, :2] / uv[:, 2:3]

def make_views(n, noise, seed, tilt=0.35, spread=60):
    rng = np.random.default_rng(seed); out = []
    for _ in range(n):
        R = rodrigues(rng.uniform(-tilt, tilt, 3))
        t = np.array([rng.uniform(-spread, 0) - 30, rng.uniform(-spread, 0) - 20, rng.uniform(380, 500)])
        out.append((project(K_TRUE, R, t, BOARD3) + rng.normal(scale=noise, size=(len(BOARD), 2)), R, t))
    return out

views = make_views(6, 0.3, 0)
for uv, _, _ in views:
    plt.plot(uv[:, 0], uv[:, 1], ".", ms=3)
plt.xlim(0, 640); plt.ylim(480, 0); plt.gca().set_aspect("equal"); plt.title("6 个视图的角点"); plt.show()

In [ ]:
def calibrate(views):
    Hs = [mod.homography_dlt(BOARD, uv) for uv, _, _ in views]
    return mod.intrinsics_from_homographies(Hs)

K = attempt(calibrate, views)
if K is not None:
    np.set_printoptions(precision=2, suppress=True)
    print("估计 K：\n", K)
    print("真值 K：\n", K_TRUE)

## 实验 1：噪声与视图数

预测：视图从 3 张增加到 15 张，fx 误差大致怎么变？

In [ ]:
res = {}
for noise in (0.1, 0.3, 1.0):
    for n in (3, 5, 10, 15):
        errs = []
        for s in range(20):
            K = attempt(calibrate, make_views(n, noise, s))
            if K is None: break
            errs.append(abs(K[0, 0] - 820) / 820)
        if not errs: break
        res[(noise, n)] = np.median(errs)
if res:
    for noise in (0.1, 0.3, 1.0):
        ns = [n for n in (3, 5, 10, 15) if (noise, n) in res]
        plt.plot(ns, [100 * res[(noise, n)] for n in ns], "o-", label=f"噪声 {noise} px")
    plt.xlabel("视图数"); plt.ylabel("fx 相对误差中位数 (%)"); plt.legend(); plt.show()

## 实验 2：姿态太单一会怎样

把倾斜角限制在 ±0.05 rad（几乎正对相机），观察主点和焦距。

In [ ]:
for tilt in (0.35, 0.05):
    K = attempt(calibrate, make_views(10, 0.3, 1, tilt=tilt))
    if K is None: break
    print(f"倾斜 ±{tilt} rad：fx={K[0,0]:.1f} fy={K[1,1]:.1f} cx={K[0,2]:.1f} cy={K[1,2]:.1f}")

## 实验 3：重投影误差小不等于标定准

In [ ]:
views = make_views(10, 0.3, 2)
K = attempt(calibrate, views)
if K is not None:
    rmse = []
    for uv, _, _ in views:
        H = mod.homography_dlt(BOARD, uv)
        R, t = mod.extrinsics_from_homography(K, H)
        rmse.append(mod.reprojection_rmse(K, R, t, BOARD3, uv))
    print("各视图重投影 RMSE：", np.round(rmse, 3))
    print("想一想：这些误差是用估计的 K 自洽地算出来的，它能发现哪些错误，发现不了哪些？")
if cv2 is not None:
    obj = [BOARD3.astype(np.float32)] * len(views)
    img = [uv.astype(np.float32) for uv, _, _ in views]
    rms, Kcv, dist, _, _ = cv2.calibrateCamera(obj, img, (640, 480), None, None)
    print("OpenCV RMS", round(rms, 3), "\nOpenCV K\n", Kcv.round(2))

## 破坏实验与记录

- 把某张视图的角点顺序打乱，结果怎样？
- 把棋盘格边长写错（15 写成 10），哪些参数受影响，哪些不受影响？
- 把实验 1 的曲线和实验 2 的结论写进 `实验记录.md`。